# 04 — FL Phase 1 Analysis (Clean Training)
Loads `round_results.csv` and `trust_scores.csv` from Phase 1 (0% attackers, 30 rounds) and visualises convergence.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sys.path.insert(0, str(Path().resolve().parent))
from src.configs.paths import RESULTS_DIR, ARTIFACTS_DIR

df = pd.read_csv(RESULTS_DIR / 'round_results.csv')
print(f'Loaded {len(df)} rounds of results.')
df.head()

In [ ]:
# Macro F1 and Accuracy vs Round
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(df['round'], df['macro_f1'], marker='o', markersize=4, color='steelblue')
ax1.set_xlabel('FL Round')
ax1.set_ylabel('Macro F1-Score')
ax1.set_title('Macro F1-Score vs Round (Phase 1 — Clean)')
ax1.axhline(0.7463, color='red', linestyle='--', label='Centralized baseline (0.7463)')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(df['round'], df['accuracy'], marker='o', markersize=4, color='green')
ax2.set_xlabel('FL Round')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy vs Round (Phase 1 — Clean)')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'plots' / 'phase1_convergence.png', dpi=150)
plt.show()
print(f'Final Macro F1: {df["macro_f1"].iloc[-1]:.4f}  |  Final Acc: {df["accuracy"].iloc[-1]:.4f}')

In [ ]:
# FPR vs Round
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df['round'], df['fpr'], marker='s', markersize=4, color='orange')
ax.set_xlabel('FL Round')
ax.set_ylabel('False Positive Rate')
ax.set_title('FPR vs Round (Phase 1 — Clean)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'plots' / 'phase1_fpr.png', dpi=150)
plt.show()

In [ ]:
# Trust score heatmap across rounds
trust_path = RESULTS_DIR / 'trust_scores.csv'
if trust_path.exists():
    trust_df = pd.read_csv(trust_path)
    pivot = trust_df.pivot(index='client_id', columns='round', values='trust_score')
    fig, ax = plt.subplots(figsize=(16, 10))
    sns.heatmap(pivot, cmap='RdYlGn', center=0, linewidths=0.2, ax=ax)
    ax.set_title('Client EMA Trust Scores per Round (Phase 1 — Clean)')
    ax.set_xlabel('FL Round')
    ax.set_ylabel('Client ID')
    plt.tight_layout()
    plt.savefig(ARTIFACTS_DIR / 'plots' / 'phase1_trust_heatmap.png', dpi=150)
    plt.show()
else:
    print('trust_scores.csv not found — run training_pipeline.py first.')

In [ ]:
# Summary stats
print('=== Phase 1 Summary ===')
print(f'Rounds:         {len(df)}')
print(f'Final Macro F1: {df["macro_f1"].iloc[-1]:.4f}')
print(f'Best Macro F1:  {df["macro_f1"].max():.4f}  (Round {df["macro_f1"].idxmax()+1})')
print(f'Final Accuracy: {df["accuracy"].iloc[-1]:.4f}')
print(f'Final FPR:      {df["fpr"].iloc[-1]:.4f}')